# nanoasr: Train a Conformer-CTC Speech Recognizer

Train a small Conformer-CTC model on LibriSpeech from scratch.

**Runtime**: Use a GPU runtime (`Runtime > Change runtime type > T4 GPU`).

In [ ]:
import os

# Private repo: store a GitHub Personal Access Token in Colab Secrets
# (key icon in left sidebar) with the name GITHUB_TOKEN.
# Create one at: github.com > Settings > Developer settings > Personal access tokens
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    !pip install -q git+https://{token}@github.com/Vaibhavdixit02/nanoasr.git
except Exception:
    # Public repo or running locally
    !pip install -q git+https://github.com/Vaibhavdixit02/nanoasr.git

In [ ]:
import torch
import torchaudio
import matplotlib.pyplot as plt
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Inspect the data pipeline

Load a few LibriSpeech samples and visualize the mel spectrogram features.

In [ ]:
import os
from nanoasr.data import LibriSpeechDataset
from nanoasr.vocab import decode_indices

os.makedirs("./data", exist_ok=True)
ds = LibriSpeechDataset(root="./data", split="dev-clean")
print(f"{len(ds)} utterances")

In [ ]:
mel, tokens = ds[0]
text = decode_indices(tokens.tolist())
print(f"Text: {text}")
print(f"Mel shape: {mel.shape}  (n_mels, T)")
print(f"Token length: {len(tokens)}")

# play the audio
waveform, sr, *_ = ds.dataset[0]
display(Audio(waveform.squeeze().numpy(), rate=sr))

# plot mel spectrogram
fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(mel.numpy(), aspect="auto", origin="lower")
ax.set_xlabel("Time frames")
ax.set_ylabel("Mel bins")
ax.set_title(f'"{text}"')
plt.tight_layout()
plt.show()

## 2. Model

The `depth` parameter controls everything: `d_model = depth * 32`, `n_heads = depth`, `n_layers = depth`.

In [ ]:
from nanoasr.model import Conformer, get_config

depth = 4
config = get_config(depth)
model = Conformer(config).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(config)
print(f"{n_params:,} parameters")

## 3. Train

In [ ]:
from nanoasr.train import train

model, config = train(
    depth=4,
    data="dev-clean",
    data_root="./data",
    epochs=50,
    batch_size=8,
    num_workers=2,
)

## 4. Save checkpoint to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil
ckpt_name = f"model_depth{depth}.pt"
drive_dest = f"/content/drive/MyDrive/{ckpt_name}"
shutil.copy(ckpt_name, drive_dest)
print(f"Saved {ckpt_name} -> {drive_dest}")
    mel, tokens = ds[i]
    ref = decode_indices(tokens.tolist())
    with torch.no_grad():
        log_probs = model(mel.unsqueeze(0).to(device))  # [1, T//4, 28]
    hyp = greedy_decode(log_probs[0].cpu())

    waveform, sr, *_ = ds.dataset[i]
    display(Audio(waveform.squeeze().numpy(), rate=sr))
    print(f"REF: {ref}")
    print(f"HYP: {hyp}")
    print()

## 5. Decode samples

Run greedy CTC decode on a few utterances.

In [ ]:
from nanoasr.decode import greedy_decode

model.eval()
for i in range(5):
    mel, tokens = ds[i]
    ref = decode_indices(tokens.tolist())
    with torch.no_grad():
        log_probs = model(mel.unsqueeze(0).to(device))  # [1, T//4, 28]
    hyp = greedy_decode(log_probs[0].cpu())

    waveform, sr, *_ = ds.dataset[i]
    display(Audio(waveform.squeeze().numpy(), rate=sr))
    print(f"REF: {ref}")
    print(f"HYP: {hyp}")
    print()